<a href="https://colab.research.google.com/github/fidlarsyn/grokking-Deep-Learning-Chapter-11-to-15/blob/main/11_Neural_Networks_That_Understand_Language.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **What Does It Mean to Understand Language?**

Memahami bahasa melalui *deep learning* berarti memodelkan data teks untuk membuat berbagai jenis prediksi. Berbeda dengan data gambar atau sinyal numerik sederhana, data teks memerlukan metode pelatihan khusus sesuai dengan karakteristik bahasa manusia.

# **Natural Language Processing (NLP)**

NLP adalah bidang ilmu yang berfokus pada otomatisasi pemahaman bahasa manusia. Secara umum, tugas-tugas dalam NLP terbagi menjadi tiga kategori utama: menguji/menandai wilayah teks (seperti *sentiment classification* atau *part-of-speech tagging*), menghubungkan dua wilayah teks (*coreference*), serta mengisi informasi teks yang hilang (*fill-in-the-blank*). Dua algoritma neural utama yang merevolusi NLP adalah *neural word embeddings* dan *Recurrent Neural Networks* (RNN).

# **Supervised NLP**

*Supervised learning* dalam NLP adalah proses mentransformasikan masukan teks ("apa yang Anda ketahui") menjadi keluaran prediksi ("apa yang ingin Anda ketahui"). Karena jaringan saraf tiruan hanya dapat memproses angka, langkah pertama yang paling krusial adalah mengonversi data teks mentah menjadi representasi matriks angka yang menonjolkan korelasi input/output.

# **IMDB Movie Reviews Dataset**

Studi kasus utama bab ini menggunakan dataset ulasan film IMDB yang berisi sekitar 50.000 pasangan ulasan teks dan rating bintang (yang dikonversi menjadi biner 0 atau 1 untuk klasifikasi sentimen positif/negatif). Tantangan utamanya adalah teks mentah memiliki panjang yang bervariasi (*variable-length text*), padahal jaringan saraf membutuhkan matriks input berukuran tetap.

In [ ]:
import sys

# 1. Membaca file ulasan dan label
f = open('reviews.txt')
raw_reviews = f.readlines()
f.close()

f = open('labels.txt')
raw_labels = f.readlines()
f.close()

# 2. Tokenisasi dan pembentukan Vocab
tokens = list(map(lambda x: set(x.split(" ")), raw_reviews))
vocab = set()
for sent in tokens:
    for word in sent:
        if len(word) > 0:
            vocab.add(word)
vocab = list(vocab)

word2index = {\}
for i, word in enumerate(vocab):
    word2index[word] = i

# 3. Konversi ulasan teks ke indeks numerik
input_dataset = list()
for sent in tokens:
    sent_indices = list()
    for word in sent:
        try:
            sent_indices.append(word2index[word])
        except:
            ""
    input_dataset.append(list(set(sent_indices)))

# 4. Konversi label positif/negatif ke 1/0
target_dataset = list()
for label in raw_labels:
    if label == 'positive\\n':
        target_dataset.append(1)
    else:
        target_dataset.append(0)

# **Capturing Word Correlation in Input Data (Bag of Words)**

Solusi untuk masalah panjang teks bervariasi adalah pendekatan **Bag of Words**. Setiap ulasan diwakili oleh vektor biner berukuran sebesar total kosakata (*vocabulary size*, misal 2.000 kata). Kolom bernilai `1` menunjukkan kehadiran kata tersebut dalam ulasan, sedangkan `0` menunjukkan ketiadaannya, tanpa memperhitungkan urutan kata.

# **Predicting Movie Reviews**

Menggunakan representasi biner Bag of Words untuk memprediksi sentimen ulasan film. Tahap pra-pemrosesan melibatkan pembagian teks menjadi token kata, pembentukan kamus pemetaan kata ke indeks (*vocabulary* &amp; `word2index`), serta konversi dataset ulasan dan label ke dalam bentuk numerik.

# **Intro to an Embedding Layer**

Melakukan perkalian matriks antara vektor *one-hot* yang sangat jarang (*sparse*) berukuran 70.000 kata dengan matriks bobot `weights_0_1` sangat tidak efisien karena sebagian besar operasi mengalikan angka 0. **Embedding Layer** berfungsi sebagai pintasan (*shortcut*): alih-alih melakukan perkalian matriks padat, jaringan langsung mengambil baris matriks bobot yang sesuai dengan indeks kata dalam ulasan dan menjumlahkannya (`np.sum(weights_0_1[x], axis=0)`).

In [ ]:
import numpy as np

np.random.seed(1)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

alpha, iterations = (0.01, 2)
hidden_size = 100

# Inisialisasi Matriks Embedding (weights_0_1) dan Matriks Output (weights_1_2)
weights_0_1 = 0.2 * np.random.random((len(vocab), hidden_size)) - 0.1
weights_1_2 = 0.2 * np.random.random((hidden_size, 1)) - 0.1

correct, total = (0, 0)
for iter in range(iterations):
    # Melatih pada 24.000 ulasan pertama
    for i in range(len(input_dataset) - 1000):
        x, y = (input_dataset[i], target_dataset[i])

        # Shortcut Embedding Layer: Lookup baris & Penjumlahan
        layer_1 = sigmoid(np.sum(weights_0_1[x], axis=0))
        layer_2 = sigmoid(np.dot(layer_1, weights_1_2))

        # Perhitungan Eror & Delta
        layer_2_error = np.sum((layer_2 - y) ** 2)
        layer_2_delta = (layer_2 - y) * layer_2 * (1 - layer_2)

        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * layer_1 * (1 - layer_1)

        # Pembaruan Bobot (Hanya baris x pada weights_0_1 yang diperbarui)
        weights_1_2 -= alpha * layer_1.reshape(-1, 1).dot(layer_2_delta)
        weights_0_1[x] -= alpha * layer_1_delta

# **Interpreting the Output**

Setelah dilatih, jaringan mencari korelasi langsung antara keberadaan kata dengan label sentimen. Kata-kata seperti *"terrible"* secara otomatis berkorelasi negatif dengan rating. Namun, jaringan dasar ini belum dapat memahami konteks bahasa yang lebih kompleks seperti negasi (misalnya perbedaan antara *"great"* dan *"not great"*).

# **Neural Architecture &amp; Grouping**

Pemilihan arsitektur menentukan apa yang dapat dipelajari oleh jaringan. *Hidden layer* berfungsi mengelompokkan masukan. Kata-kata yang memiliki kekuatan prediktif serupa (sama-sama memprediksi positif atau negatif) akan dikelompokkan ke dalam nodus *hidden layer* yang mirip, sehingga nilai bobot yang terhubung ke nodus tersebut akan bernilai serupa.

# **Comparing Word Embeddings**

Vektor bobot baris `weights_0_1` untuk suatu kata membentuk **word embedding** dari kata tersebut. Dengan menghitung **Jarak Euclidean** (*Euclidean Distance*) antar-baris vektor bobot kata, kita dapat menemukan kata-kata yang paling serupa dalam matriks bobot. Pada model sentimen, kata *"terrible"* akan memiliki vektor yang sangat mirip dengan kata *"dull"*, *"boring"*, dan *"horrible"* karena semuanya memprediksi label negatif.

In [ ]:
import math
from collections import Counter

def similar(target='beautiful'):
    target_index = word2index[target]
    scores = Counter()
    for word, index in word2index.items():
        raw_difference = weights_0_1[index] - (weights_0_1[target_index])
        squared_difference = raw_difference * raw_difference
        scores[word] = -math.sqrt(sum(squared_difference))
    return scores.most_common(10)

# Contoh Penggunaan:
# print(similar('beautiful'))
# print(similar('terrible'))

# **What Is the Meaning of a Neuron?**

Makna (*meaning*) dari suatu neuron atau *word embedding* sepenuhnya ditentukan oleh **target label (fungsi kerugian / loss function)** yang diprediksi oleh jaringan. Pada model sentimen, kata *"beautiful"* paling mirip dengan *"atmosphere"* karena keduanya sama-sama memprediksi ulasan positif, bukan karena memiliki jenis tata bahasa yang sama.

# **Filling in the Blank (Self-Supervised Learning)**

Untuk memaksa jaringan mempelajari makna kata yang lebih kaya dan bernuansa semantik, kita dapat mengubah tugas prediksinya menjadi tugas **fill-in-the-blank** (memprediksi kata yang dihilangkan dari jendela konteks kalimat). Untuk mempercepat pelatihan dari ribuan label kata, digunakan teknik **Negative Sampling** (hanya memprediksi target benar dan beberapa sampel kata acak yang diabaikan).

In [ ]:
import numpy as np
import sys, random, math
from collections import Counter

np.random.seed(1)
random.seed(1)

f = open('reviews.txt')
raw_reviews = f.readlines()
f.close()

tokens = list(map(lambda x: (x.split(" ")), raw_reviews))
wordcnt = Counter()
for sent in tokens:
    for word in sent:
        wordcnt[word] -= 1
vocab = list(set(map(lambda x: x, wordcnt.most_common())))

word2index = {\}
for i, word in enumerate(vocab):
    word2index[word] = i

concatenated = list()
input_dataset = list()
for sent in tokens:
    sent_indices = list()
    for word in sent:
        try:
            sent_indices.append(word2index[word])
            concatenated.append(word2index[word])
        except:
            ""
    input_dataset.append(sent_indices)

concatenated = np.array(concatenated)
random.shuffle(input_dataset)

alpha, iterations = (0.05, 2)
hidden_size, window, negative = (50, 2, 5)

weights_0_1 = (np.random.rand(len(vocab), hidden_size) - 0.5) * 0.2
weights_1_2 = np.random.rand(len(vocab), hidden_size) * 0

layer_2_target = np.zeros(negative + 1)
layer_2_target = 1

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def similar(target='beautiful'):
    target_index = word2index[target]
    scores = Counter()
    for word, index in word2index.items():
        raw_difference = weights_0_1[index] - (weights_0_1[target_index])
        squared_difference = raw_difference * raw_difference
        scores[word] = -math.sqrt(sum(squared_difference))
    return scores.most_common(10)

# Training Loop Fill-in-the-blank + Negative Sampling
for rev_i, review in enumerate(input_dataset * iterations):
    for target_i in range(len(review)):
        target_samples = [review[target_i]] + list(concatenated[(np.random.rand(negative) * len(concatenated)).astype('int').tolist()])
        left_context = review[max(0, target_i - window):target_i]
        right_context = review[target_i + 1:min(len(review), target_i + window)]

        layer_1 = np.mean(weights_0_1[left_context + right_context], axis=0)
        layer_2 = sigmoid(layer_1.dot(weights_1_2[target_samples].T))
        layer_2_delta = layer_2 - layer_2_target
        layer_1_delta = layer_2_delta.dot(weights_1_2[target_samples])

        weights_0_1[left_context + right_context] -= layer_1_delta * alpha
        weights_1_2[target_samples] -= np.outer(layer_2_delta, layer_1) * alpha

    if (rev_i % 250 == 0):
        sys.stdout.write('\\rProgress:' + str(rev_i / float(len(input_dataset) * iterations)) + " " + str(similar('terrible')))

# **Meaning Is Derived from Loss**

Dengan menggeser tugas prediksi dari klasifikasi sentimen ke *fill-in-the-blank*, kelompok *word embedding* berubah secara drastis. Pada tugas *fill-in-the-blank*, kata *"terrible"* kini paling mirip dengan *"horrible"*, *"brilliant"*, dan *"pathetic"* karena kata-kata tersebut muncul pada konteks frasa kalimat yang serupa. Hal ini membuktikan prinsip *intelligence targeting*: kita dapat mengarahkan apa yang dipelajari jaringan dengan memilih nilai target dan fungsi kerugian.

# **King – Man + Woman \~= Queen (Word Analogies)**

 Konsekuensi menarik dari word embeddings yang dilatih pada tugas fill-in-the-blank adalah munculnya sifat analogi kata. Kita dapat melakukan operasi aritmatika linier pada vektor kata, seperti $\text{king} - \text{man} + \text{woman} \approx \text{queen}$. Hal ini terjadi karena kata "king" merepresentasikan dimensi royalty dan male, sehingga mengurangkan vektor man dan menambahkan vektor woman menghasilkan posisi vektor yang paling dekat dengan "queen".

In [ ]:
import math
from collections import Counter
import numpy as np

def analogy(positive=['terrible', 'good'], negative=['bad']):
    norms = np.sum(weights_0_1 * weights_0_1, axis=1)
    norms.resize(norms.shape, 1)
    normed_weights = weights_0_1 * norms

    query_vect = np.zeros(len(weights_0_1))
    for word in positive:
        query_vect += normed_weights[word2index[word]]
    for word in negative:
        query_vect -= normed_weights[word2index[word]]

    scores = Counter()
    for word, index in word2index.items():
        raw_difference = weights_0_1[index] - query_vect
        squared_difference = raw_difference * raw_difference
        scores[word] = -math.sqrt(sum(squared_difference))
    return scores.most_common(10)[1:]

# Contoh penggunaan analogi:
# analogy(positive=['king', 'woman'], negative=['man']) -> menghasilkan 'queen'